In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import kagglehub
import os
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm

In [ ]:
# Create a custom Dataset class or use ImageFolder
transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),

])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])


# Create dataset objects
dataset_path = os.path.join(path, "q1-stage-3-2026", "PlantVillage","train")
# Get all image paths
train_image_paths = glob.glob(f"{dataset_path}/*.jpg")

dataset_path = os.path.join(path, "q1-stage-3-2026", "PlantVillage","test")
# Get all image paths
test_image_paths = glob.glob(f"{dataset_path}/*.jpg")

train_labels = []
test_labels= []

for path in tqdm(train_image_paths):
    prefix = path.split("___")[-1].split(" ")[0]

    if(prefix=="Early_blight"):
      label = 0
      train_labels.append(label)

    if(prefix=="Late_blight"):
      label = 1
      train_labels.append(label)

    if(prefix=="healthy"):
      label = 2
      train_labels.append(label)

for path in tqdm(test_image_paths):
    prefix = path.split("___")[-1].split(" ")[0]

    if(prefix=="Early_blight"):
      label = 0
      test_labels.append(label)

    if(prefix=="Late_blight"):
      label = 1
      test_labels.append(label)

    if(prefix=="healthy"):
      label = 2
      test_labels.append(label)



In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class CustomDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths  # List of image paths
        self.labels = labels  # Corresponding labels
        self.transform = transform  # Transformations to apply

    def __len__(self):
        return len(self.image_paths)  # Total number of images

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]  # Get image path
        label = self.labels[idx]  # Get corresponding label

        # Load image
        image = Image.open(image_path)

        # Apply transformations (if any)
        if self.transform:
            image = self.transform(image)

        return image, label  # Return processed image and its label

In [ ]:
train_dataset = CustomDataset(train_image_paths, train_labels, transform=transform)
test_dataset = CustomDataset(test_image_paths, test_labels, transform=test_transform)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Get a batch of training images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")

In [ ]:
#displaying:
fig, axes = plt.subplots(2, 6, figsize=(10, 4)) #(2 rows, 6 col)
for ax in axes.ravel(): #flatten the 2D axes array into a 1D iterator
    idx = np.random.randint(0, len(train_ds))
    sample = train_ds[idx]
    x, y = sample["image"], sample["label"]

    # unnormalize for display (approx)
    print(x_vis.shape) #to double check the size
    ax.imshow(x_vis.squeeze(0)) #remove channel dim and show image
    ax.set_title(class_names[y], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Automatically select CPU or GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create a tensor and move it to GPU
x_gpu = x.to(device)
print(x_gpu.device)  # Output: cuda:0 (if GPU is available) or cpu


In [ ]:
#creating a CNN from scratch
import torch
import torch.nn as nn

class CustomModel(nn.Module):
    def __init__(self):
        super(CustomModel, self).__init__()
# Define layers
        nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1) #in=3 because it's RGB | out_channels are design choice
        nn.BatchNorm2d(16)
        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1) #math:floor(16+2-3/1) +1 = 16*16
        nn.BatchNorm2d(32)
        nn.Conv2d(in_channels=32, out_channels=48, kernel_size=3, stride=1, padding=1) #math:floor(48+2-3/1) +1 = 48*48
        nn.BatchNorm2d(48)
        nn.Conv2d(in_channels=48, out_channels=64, kernel_size=3, stride=1, padding=1) #same math..
        nn.BatchNorm2d(64)
        nn.Conv2d(in_channels=64, out_channels=80, kernel_size=3, stride=1, padding=1) #same math..
        nn.BatchNorm2d(80) #(64+2-3)+1 => 64*64

        self.classifier = nn.Sequential(
            nn.Flatten(),
            self.fc = nn.Linear(64 * 64 * 16, 10) )

        # Define the forward pass (how data flows through the model).
      def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()
